## Example 1: Parse a URL into its parts

In [ ]:
import furl

f = furl.furl("http://user:pw@example.com:8080/a/b?x=1&x=2&y=z#frag")
print("scheme:", f.scheme, "| host:", f.host, "| port:", f.port)
print("path segments:", f.path.segments, "| args:", dict(f.args), "| fragment:", f.fragment)
assert (f.scheme, f.username, f.password) == ("http", "user", "pw")
assert (f.host, f.port) == ("example.com", 8080)
assert f.netloc == "user:pw@example.com:8080"
assert f.origin == "http://example.com:8080"
assert f.path.segments == ["a", "b"]
assert f.args["y"] == "z"
assert f.args.getlist("x") == ["1", "2"]  # repeated keys are kept, in order
assert str(f.fragment) == "frag"

## Example 2: Build a URL step by step

In [ ]:
import furl

f = furl.furl("http://example.com/a/b?x=1")
f.args["y"] = "2"
f.path.segments.append("c")
print("built:", f.url)
assert f.url == "http://example.com/a/b/c?x=1&y=2"

# add() appends, set() replaces; both return the furl so calls chain.
g = furl.furl("http://example.com/").add(path="x/y", args={"a": "1"})
assert g.url == "http://example.com/x/y?a=1"
g.set(args={"b": "2"})
assert g.url == "http://example.com/x/y?b=2"
print("chained:", g.url)

## Example 3: Remove parts and copy

In [ ]:
import furl

f = furl.furl("http://example.com/a/b/c?x=1&y=2#top")
trimmed = f.copy().remove(["x"], fragment=True)
print("original:", f.url)
print("trimmed: ", trimmed.url)
assert trimmed.url == "http://example.com/a/b/c?y=2"
assert f.url == "http://example.com/a/b/c?x=1&y=2#top"  # copy() left the original alone

f.remove(args=True, path=True)
assert f.url == "http://example.com#top"

## Example 4: Encoding of paths, queries and hosts

In [ ]:
import furl

f = furl.furl("http://example.com").set(path="/p q", args={"k": "v w"})
print("encoded:", f.url)
assert f.url == "http://example.com/p%20q?k=v+w"  # space is %20 in a path, + in a query
assert f.path.segments == ["p q"]  # segments hold the decoded text

q = furl.Query("a=1&b=2&a=3")
assert q.params.allitems() == [("a", "1"), ("b", "2"), ("a", "3")]
assert q.encode(";") == "a=1;b=2;a=3"

# An internationalised host name is encoded with IDNA (punycode).
idn = furl.furl("http://bücher.example/")
print("IDN host:", idn.url)
assert idn.url == "http://xn--bcher-kva.example/"

## Example 5: Join relative URLs, default ports and paths

In [ ]:
import furl

base = furl.furl("http://example.com/a/b/")
assert base.copy().join("../c").url == "http://example.com/a/c"
assert furl.furl("http://example.com/a/b").join("c?d=1").url == "http://example.com/a/c?d=1"

# The port falls back to the scheme's default when the URL has none.
assert furl.furl("http://user:pw@h:8080/").port == 8080
assert furl.furl("https://h/").port == 443 and furl.furl("http://h/").port == 80

p = furl.Path("/a/b/../c")
p.normalize()
assert str(p) == "/a/c" and p.isabsolute and not p.isdir
assert furl.furl("http://example.com/x") == furl.furl("http://example.com/x")
print("join, ports and paths OK:", str(p))